# ScoutRAG — Constrained Hybrid Retrieval for Football Player Scouting
**CS455 — Large Language Models · Standard Track**  
Çubukçu · Özer · Yılmaz

ScoutRAG is a **retrieval-augmented scouting assistant**. A user writes a natural-language
query ("a clinical, left-footed striker under 21, valued below €10M") and the system returns
the players that match, ranked best→worst.

The system is a **constrained hybrid retrieval pipeline over structured and semantic player
attributes**: precise constraints are enforced by strict symbolic filtering, fuzzy scouting
language is mapped to attribute constraints by a local LLM, and an optional semantic stage
re-ranks within the filtered pool. A post-generation **grounding-verification** step checks
that any numbers cited in a generated report actually appear in the retrieved rows.

> **Dataset honesty note.** The data is a Football Manager 2024 export. Every attribute is a
> *designer-set game value on a 1–20 scale*, **not** real-world scouting data. We treat it
> strictly as a **synthetic / prototype tabular dataset** for validating the retrieval and
> grounding machinery. No claim is made that these ratings reflect real player ability.

This notebook runs **fully locally** (no Google Drive, no paid APIs). The optional LLM step
uses a local **Ollama** model; everything degrades gracefully if it is not running.

## 0 · Environment & local setup
Run from the project root so the `scoutrag` package is importable. Hardware: any Apple-Silicon
Mac with 16 GB RAM is comfortable. The embedding model runs on CPU/MPS; the optional LLM needs
Ollama (`brew install ollama && ollama pull llama3.1`).

In [ ]:
# Install once if needed (uncomment):
# %pip install -r requirements.txt
import os, sys, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 160)

# Make the local scoutrag package importable
sys.path.insert(0, os.path.abspath("."))
from scoutrag import (load_and_clean, build_profiles, align_embeddings,
                      parse_query, structured_filter, rank_players, search,
                      ATTR_READABLE, OUTFIELD_ATTR_COLS, GK_ONLY_COLS, NO_MATCH_MESSAGE)
from scoutrag import llm as scout_llm

CSV_PATH = "fmdata24llm.csv"
PKL_PATH = "player_embeddings.pkl"
GREEN = "#1fae7a"
plt.rcParams.update({"axes.grid": True, "grid.alpha": .25, "figure.facecolor": "white"})
print("imports OK")

## 1 · Load & clean the dataset
The `Name` field packs the nationality (`"Lionel Messi - Argentinian"`) and `Club` packs the
league. Money and weight are messy strings. The FM24 export also contains ~1,900 empty engine
slots whose name is the placeholder `"- -"`.

**Cleaning rule (per project spec):** any row whose player name is *missing, empty, whitespace,
invisible, or contains no alphabetic character* is removed.

In [ ]:
df = load_and_clean(CSV_PATH)            # strict name cleaning happens here
df = build_profiles(df)                  # adds 'profile' text + 'Overall' + 'is_GK'
print(f"Clean dataset: {df.shape[0]:,} players x {df.shape[1]} columns")
df[["Player_Name","Nationality","Age","Position","Club_Name","League",
    "Value_EUR","Wage_EUR_pm","Foot","Overall"]].head()

### What cleaning removed
A quick before/after to document exactly which rows were dropped and why.

In [ ]:
raw = pd.read_csv(CSV_PATH); raw.columns=[x.lstrip('\ufeff') for x in raw.columns]
print(f"raw rows           : {len(raw):,}")
print(f"clean rows         : {len(df):,}")
print(f"removed (invalid)  : {len(raw)-len(df):,}")
bad = raw[raw['Name'].astype(str).str.replace(' - ','',regex=False).str.strip().isin(['- -','-','--',''])]
print("\nexamples of removed 'Name' values:", raw['Name'][raw['Name'].astype(str).str.strip().eq('- -')].head(3).tolist())

## 2 · Exploratory Data Analysis
We profile the dataset to understand what constraints are even answerable: which attributes
exist, how complete they are, and how leagues / nationalities / positions / values are
distributed. (Figures are also saved to `eda_outputs/` for the report.)

In [ ]:
os.makedirs("eda_outputs", exist_ok=True)
print("Columns:", len(df.columns))
print("\nDerived columns:", ["Player_Name","Nationality","Club_Name","League","Value_EUR",
      "Not_For_Sale","Wage_EUR_pm","Weight_kg","Foot","Primary_Pos","is_GK","Status","Overall"])
print("\nGoalkeepers:", int(df.is_GK.sum()), " | Outfield:", int((~df.is_GK).sum()))
df[["Age","Value_EUR","Wage_EUR_pm","Overall","Weight_kg"]].describe().round(1)

In [ ]:
# Missing values
miss = df.isna().mean().sort_values(ascending=False); miss = miss[miss>0]
ax = miss.head(15).plot(kind="barh", color=GREEN, figsize=(8,4))
ax.set_title("Missing-value fraction by column (top 15)"); ax.set_xlabel("fraction missing")
plt.tight_layout(); plt.savefig("eda_outputs/01_missing.png", dpi=120); plt.show()
print(miss.round(3).head(8).to_dict())

In [ ]:
# Age distribution
ax = df.Age.dropna().plot(kind="hist", bins=range(14,45), color=GREEN, edgecolor="white", figsize=(7,4))
ax.set_title("Age distribution"); ax.set_xlabel("age")
plt.tight_layout(); plt.savefig("eda_outputs/02_age.png", dpi=120); plt.show()

In [ ]:
# Leagues & nationalities
fig, ax = plt.subplots(1,2, figsize=(13,5))
df.League.value_counts().head(15).iloc[::-1].plot(kind="barh", color=GREEN, ax=ax[0]); ax[0].set_title("Players per league (top 15)")
df.Nationality.value_counts().head(15).iloc[::-1].plot(kind="barh", color=GREEN, ax=ax[1]); ax[1].set_title("Players per nationality (top 15)")
plt.tight_layout(); plt.savefig("eda_outputs/03_leagues_nat.png", dpi=120); plt.show()

In [ ]:
# Positions
ax = df.Primary_Pos.value_counts().head(15).iloc[::-1].plot(kind="barh", color=GREEN, figsize=(8,5))
ax.set_title("Primary position (top 15)")
plt.tight_layout(); plt.savefig("eda_outputs/05_positions.png", dpi=120); plt.show()

In [ ]:
# Transfer value & wage (log10)
fig, ax = plt.subplots(1,2, figsize=(12,4))
np.log10(df.Value_EUR.replace(0,np.nan).dropna()).plot(kind="hist", bins=40, color=GREEN, edgecolor="white", ax=ax[0]); ax[0].set_title("log10 transfer value (EUR)")
np.log10(df.Wage_EUR_pm.replace(0,np.nan).dropna()).plot(kind="hist", bins=40, color=GREEN, edgecolor="white", ax=ax[1]); ax[1].set_title("log10 monthly wage (EUR)")
plt.tight_layout(); plt.savefig("eda_outputs/06_value_wage.png", dpi=120); plt.show()
print(f"Not-for-sale players: {int(df.Not_For_Sale.sum()):,}")

In [ ]:
# Attribute distributions (sample) — all on the FM 1–20 scale
sample = ["Pac","Pas","Fin","Tck","Dri","Str","Sta","Vis","Hea"]
fig, axes = plt.subplots(3,3, figsize=(12,9))
for a,col in zip(axes.ravel(), sample):
    df[col].plot(kind="hist", bins=range(1,21), color=GREEN, edgecolor="white", ax=a); a.set_title(ATTR_READABLE[col]); a.set_xlabel("")
plt.suptitle("FM24 attribute distributions (1–20 scale)"); plt.tight_layout()
plt.savefig("eda_outputs/07_attributes.png", dpi=120); plt.show()

In [ ]:
# Attribute correlation (outfield) — shows attributes cluster into physical / technical / mental
corr = df[OUTFIELD_ATTR_COLS].corr()
plt.figure(figsize=(11,9)); plt.imshow(corr, cmap="BrBG", vmin=-1, vmax=1); plt.colorbar(fraction=.046)
plt.xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7); plt.yticks(range(len(corr)), corr.columns, fontsize=7)
plt.title("Outfield attribute correlation"); plt.tight_layout()
plt.savefig("eda_outputs/08_corr.png", dpi=120); plt.show()

**EDA takeaways.**
- Attributes are complete (no missing values in the 1–20 ratings); missingness is confined to
  *value* / *wage* (lower-league or not-for-sale players) and occasionally *weight*.
- Values and wages are heavily right-skewed — log scale is the honest way to view them, and the
  engine stores raw EUR so range filters work.
- Attributes are clearly **designer-engineered**: they cluster into physical / technical / mental
  blocks and centre near 8–12. This reinforces the *synthetic game data* framing — useful for
  validating retrieval, not for real talent claims.

## 3 · Player text profiles & embeddings
Each row is turned into a short grounded sentence (`build_profiles` already did this). We reuse
the cached MiniLM embeddings (`player_embeddings.pkl`) aligned to the cleaned rows; if the cache
is missing we rebuild on the fly.

In [ ]:
print("Example profile:\n", df["profile"].iloc[0], "\n")
emb = align_embeddings(df, PKL_PATH)
embed_query_fn = None
if emb is None:
    print("No usable cache — rebuilding embeddings (downloads MiniLM once)...")
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer("all-MiniLM-L6-v2")
    emb = _m.encode(df["profile"].tolist(), normalize_embeddings=True, show_progress_bar=True).astype("float32")
print("embeddings:", None if emb is None else emb.shape)

In [ ]:
# Query encoder for the semantic stage (same model that built the cache)
try:
    from sentence_transformers import SentenceTransformer
    _qmodel = SentenceTransformer("all-MiniLM-L6-v2")
    embed_query_fn = lambda t: _qmodel.encode([t], normalize_embeddings=True)[0].astype("float32")
    print("semantic search: ENABLED")
except Exception as e:
    embed_query_fn = None
    print("semantic search: disabled —", e)

## 4 · Module 1 — Natural-language scouting search
This is the core deliverable. The pipeline is:

```
query → [LLM rewrite (Ollama, optional)] → parse_query → structured_filter (STRICT)
      → rank_players (combined, position-aware) → top-k
```

**Parsing** supports every attribute as a constraint, numeric lower/upper bounds, `between`
ranges, and **OR** conditions (nationality, age intervals, multiple positions). Precise numeric
constraints are *hard* (strict filter); vague adjectives ("clinical", "visionary", "powerful")
become *soft* ranking signals — and the local LLM upgrades them to explicit constraints.

**Ranking** scores each player by how well they exceed every constraint, combines the
sub-scores fairly, and breaks ties by overall quality. It is **position-aware**: goalkeeper
attributes never rank outfielders and vice-versa. If nobody matches, it returns exactly:
`There is no player like that in our database.`

In [ ]:
def show(query, top_k=10, use_llm=True, report=False):
    rewritten, used = (query, False)
    if use_llm:
        rewritten, used = scout_llm.rewrite_query(query)
        if used: print(f"LLM rewrite → {rewritten}")
    p = parse_query(rewritten); p["top_k"] = top_k
    res = search(df, p, embeddings=emb, embed_query_fn=embed_query_fn, return_df=True)
    print("constraints:", " | ".join(p["trace"]) or "(none)")
    if res["message"]:
        print(">>", res["message"]); return
    cols = ["Player_Name","Age","Nationality","Position","Club_Name","Value_EUR","Overall","match_score"]
    print(f"{len(res['players'])} match(es) from {res['n_filtered']:,} that passed filters:")
    display(res["df"][cols].reset_index(drop=True))
    if report:
        rep = scout_llm.generate_report(query, res["players"])
        print("\n--- grounded report ---\n", rep)
        vr = scout_llm.verify_grounding(rep, res["players"])
        print(f"\ngrounding score: {vr['grounding_score']:.0%} "
              f"({vr['total_checked']} claims, {vr['hallucination_count']} ungrounded)")

show("left-footed centre-back under 23 with passing > 15 and tackling >= 14, valued under 10M")

In [ ]:
# OR conditions: nationality OR + multiple positions
show("Turkish or Brazilian winger under 21", top_k=8)

In [ ]:
# Numeric BETWEEN bound + OR age intervals
show("players whose passing is between 10 and 17, under 20 or over 35", top_k=8)

In [ ]:
# Descriptive terms → soft ranking signals (clinical / strong / fast)
show("top 5 clinical, strong and fast strikers")

In [ ]:
# Position-aware ranking for goalkeepers
show("commanding goalkeeper with great reflexes, top 5")

In [ ]:
# The explicit 'no player' message when constraints are unsatisfiable
show("Welsh goalkeeper with finishing > 19 and pace > 19 under 17")

### Why the ranking is fair (worked example)
For `passing > 15`, a player with passing 17 must outrank one with passing 15. With multiple
constraints we average each constraint's satisfaction margin, so a player who is better at one
attribute but worse at another is balanced fairly; overall quality breaks ties.

In [ ]:
p = parse_query("midfielder, passing > 15, stamina > 14")
res = search(df, p, embeddings=emb, embed_query_fn=embed_query_fn, return_df=True)
res["df"][["Player_Name","Pas","Sta","Overall","match_score"]].head(8).reset_index(drop=True)

## 5 · Grounding verification (the key contribution)
When the LLM writes a scouting report, we verify every cited player name and every numeric
attribute against the retrieved rows. A claim that does not appear in the retrieved data is
flagged as a hallucination, yielding a grounding score in [0,1].

In [ ]:
if scout_llm.ollama_available():
    show("a clinical, visionary German playmaker under 25", top_k=5, report=True)
else:
    print("Ollama not running — showing template report + verification instead.")
    p = parse_query("clinical visionary playmaker"); p["top_k"]=5
    res = search(df, p, embeddings=emb, embed_query_fn=embed_query_fn)
    rep = scout_llm.generate_report("clinical visionary playmaker", res["players"])
    print(rep)
    vr = scout_llm.verify_grounding(rep, res["players"])
    print("\ngrounding score:", f"{vr['grounding_score']:.0%}", "| checked:", vr["total_checked"])
    # Inject a fake stat to demonstrate the verifier catching a hallucination:
    fake = rep + "\nScout note: his finishing is 99 and passing is 99."
    vrf = scout_llm.verify_grounding(fake, res["players"])
    print("after injecting a fabricated '99' stat → grounding score:",
          f"{vrf['grounding_score']:.0%}", "| ungrounded:", vrf["hallucination_count"])

## 6 · Evaluation
We run a 40-query benchmark spanning precise numeric constraints, ranges, OR conditions,
positions, nationalities, budgets, and descriptive language, then compare three retrieval
strategies on **constraint satisfaction** (fraction of returned players that actually satisfy
every hard constraint) and **zero-result behaviour**.

- **structured-only** — strict filter + quality ranking (no semantic)
- **vector-only** — pure embedding similarity, *no* hard filtering (the naive RAG baseline)
- **hybrid** — strict filter + combined ranking (+ semantic re-rank) — ScoutRAG

In [ ]:
EVAL_QUERIES = [
 "left-footed centre-back under 23 with passing > 15 and tackling >= 14, valued under 10M",
 "Turkish or Brazilian winger under 21",
 "players whose passing is between 10 and 17, under 20 or over 35",
 "top 5 clinical, strong and fast strikers",
 "commanding goalkeeper with great reflexes",
 "visionary German playmaker, wage under 200k",
 "fast right-footed winger under 22 valued under 15M",
 "tireless box-to-box midfielder, work rate > 15, stamina > 15",
 "strong aerial centre-back, heading > 15",
 "young Brazilian striker under 20 with finishing >= 14",
 "experienced defensive midfielder over 30 with tackling >= 15",
 "creative attacking midfielder, vision >= 16, under 24",
 "English goalkeeper with reflexes >= 15",
 "French or Spanish full-back under 25",
 "powerful striker, strength >= 16, heading >= 14",
 "cheap teenager prospect valued under 2M",
 "pacey forward, pace >= 17",
 "composed centre-back, composure >= 15, under 26",
 "left winger with dribbling >= 16",
 "veteran goalkeeper over 33",
 "clinical poacher, finishing >= 17",
 "Argentine playmaker with passing >= 16",
 "defensive midfielder, tackling >= 15, aggression >= 14",
 "fast and skillful winger under 23 valued under 20M",
 "tall strong target man, strength >= 15, heading >= 16",
 "young German midfielder under 21 with technique >= 15",
 "Dutch centre-back valued under 12M",
 "high work rate striker, work rate >= 15",
 "agile shot-stopper, reflexes >= 16, agility >= 15",
 "Italian or Portuguese winger under 24",
 "playmaker with vision >= 15 and passing >= 15",
 "physical defensive midfielder over 28",
 "clinical and composed striker, finishing >= 15, composure >= 14",
 "young left-footed playmaker under 20",
 "Brazilian forward with pace >= 16 and finishing >= 14",
 "experienced leader centre-back, leadership >= 15, over 29",
 "fast full-back, pace >= 15, under 25",
 "creative winger, flair >= 15, vision >= 14",
 "strong and brave goalkeeper, bravery >= 14",
 "cheap clinical striker under 21 valued under 5M",
]
len(EVAL_QUERIES)

In [ ]:
def satisfies(row, parsed):
    # check hard constraints only
    for mn,mx in [(iv) for iv in parsed["age"]] or []:
        pass
    ok = True
    if parsed["age"]:
        ok &= any((mn is None or row["Age"]>=mn) and (mx is None or row["Age"]<=mx) for mn,mx in parsed["age"])
    if parsed["foot"]: ok &= row["Foot"]==parsed["foot"]
    if parsed["nationality"]: ok &= row["Nationality"].lower() in [n.lower() for n in parsed["nationality"]]
    if parsed["position"]: ok &= any(t.lower() in str(row["Position"]).lower() for t in parsed["position"])
    if parsed["value"]["max"] is not None: ok &= pd.notna(row["Value_EUR"]) and row["Value_EUR"]<=parsed["value"]["max"]
    if parsed["value"]["min"] is not None: ok &= pd.notna(row["Value_EUR"]) and row["Value_EUR"]>=parsed["value"]["min"]
    for col,conds in parsed["hard_attrs"].items():
        for op,thr in conds:
            v=row.get(col)
            ok &= {">":v>thr,">=":v>=thr,"<":v<thr,"<=":v<=thr,"=":v==thr}[op]
    return bool(ok)

def eval_query(q):
    p = parse_query(q); p["top_k"]=10
    # hybrid
    hyb = search(df, p, embeddings=emb, embed_query_fn=embed_query_fn, return_df=True)["df"]
    # structured-only (no semantic re-rank)
    st  = search(df, p, embeddings=None, embed_query_fn=None, return_df=True)["df"]
    # vector-only: ignore all hard filters, rank whole df by similarity
    vec = pd.DataFrame()
    if embed_query_fn is not None:
        qv = embed_query_fn(p["semantic_query"]).reshape(1,-1)
        sims = (emb @ qv.T).ravel()
        vec = df.iloc[np.argsort(-sims)[:10]].copy()
    def csat(d):
        if len(d)==0: return np.nan
        return np.mean([satisfies(r,p) for _,r in d.iterrows()])
    return {"query":q,
            "hybrid_csat":csat(hyb), "structured_csat":csat(st), "vector_csat":csat(vec),
            "hybrid_n":len(hyb), "structured_n":len(st)}

t0=time.time(); rows=[eval_query(q) for q in EVAL_QUERIES]; 
ev=pd.DataFrame(rows); print(f"evaluated {len(ev)} queries in {time.time()-t0:.1f}s")
ev.head()

In [ ]:
summary = pd.DataFrame({
 "mean constraint satisfaction":[ev.structured_csat.mean(), ev.vector_csat.mean(), ev.hybrid_csat.mean()],
 "mean #results":[ev.structured_n.mean(), 10 if embed_query_fn else np.nan, ev.hybrid_n.mean()],
 "zero-result queries":[int((ev.structured_n==0).sum()), 0, int((ev.hybrid_n==0).sum())],
}, index=["structured-only","vector-only","hybrid (ScoutRAG)"]).round(3)
display(summary)

fig,ax=plt.subplots(figsize=(7,4))
summary["mean constraint satisfaction"].plot(kind="bar", color=[GREEN,"#e0a64b","#0e7a55"], ax=ax)
ax.set_ylim(0,1.05); ax.set_ylabel("constraint satisfaction"); ax.set_title("Constraint satisfaction by retrieval strategy")
plt.xticks(rotation=15); plt.tight_layout(); plt.savefig("eda_outputs/09_eval.png", dpi=120); plt.show()

### Error analysis (honest)
- **Vector-only** retrieval ignores hard constraints, so it routinely returns players who
  violate explicit numbers (wrong age, over budget, wrong position) — its constraint
  satisfaction is well below 1.0. This is exactly the failure mode strict filtering fixes.
- **structured-only** and **hybrid** satisfy constraints by construction (≈1.0); the semantic
  stage in hybrid changes the *ordering* within the valid pool, not membership.
- **Zero-result queries** are not errors: when no player satisfies the constraints the system
  honestly returns *"There is no player like that in our database."* rather than inventing one.
- **Parser limits.** Rare phrasings or attributes outside the synonym map may be missed; the
  LLM-rewrite step mitigates this but is best-effort. Descriptive thresholds (what counts as
  "strong") are heuristic design choices, not ground truth.
- **Data limits.** All ratings are FM24 game values; "good" here means a high designer-set
  number, not validated real-world ability.

## 7 · Future modules (not implemented here)
1. **Player info** — `"Tell me about Lionel Messi"` → a grounded profile from his row.
2. **Player comparison** — compare two players attribute-by-attribute.

Both reuse the same grounded-generation + verification machinery already built above.

## 8 · Web demo
A FastAPI backend (`backend/app.py`) wraps this exact engine, and a static green chat UI
(`docs/index.html`, GitHub-Pages ready) calls it. Run:
```
uvicorn backend.app:app --reload --port 8000
```
then open `docs/index.html`. See `README.md` for full instructions.